# Nível 2 - Parte A: Regras em escala

Este notebook reaproveita as decisões validadas no Nível 1: base bruta preservada, remoção apenas de duplicatas exatas, datas ausentes mantidas, conversão monetária em coluna derivada e regras calculadas somente com Pandas. Nenhuma LLM participa desta etapa.


In [1]:
import json
from pathlib import Path

import pandas as pd


## Carregamento e base bruta

A base bruta permanece inalterada. A taxa de câmbio é lida exclusivamente do próprio JSON.


In [2]:
caminho_dados = Path('../dados/dados_nivel_2.json')

with caminho_dados.open(encoding='utf-8') as arquivo:
    dados_brutos = json.load(arquivo)

taxa_cambio_usd_brl = dados_brutos['taxa_cambio_usd_brl']
df_operacoes_bruto = pd.DataFrame(dados_brutos['operacoes'])

print(f'Operações brutas: {len(df_operacoes_bruto)}')
print(f'Clientes únicos: {df_operacoes_bruto["cliente_id"].nunique()}')
print(f'Taxa USD/BRL: {taxa_cambio_usd_brl}')


Operações brutas: 322
Clientes únicos: 30
Taxa USD/BRL: 5.4


## Limpeza e normalização monetária

Somente duplicatas de linhas inteiras são removidas. A base bruta tem sete datas ausentes; após a deduplicação, seis operações únicas continuam sem data, pois uma das duplicatas removidas também tinha data ausente. Nenhuma linha é removida exclusivamente por ausência de data ou por observação ausente.


In [3]:
df_operacoes_tratado = df_operacoes_bruto.drop_duplicates().copy()

print(f'Operações após deduplicação: {len(df_operacoes_tratado)}')


Operações após deduplicação: 317


In [4]:
df_operacoes_tratado['data'] = pd.to_datetime(df_operacoes_tratado['data'])

print(f'Registros sem data preservados: {df_operacoes_tratado["data"].isna().sum()}')


Registros sem data preservados: 6


In [5]:
moedas_esperadas = {'BRL', 'USD'}
moedas_encontradas = set(df_operacoes_tratado['moeda'].unique())
moedas_inesperadas = moedas_encontradas - moedas_esperadas

if moedas_inesperadas:
    raise ValueError(
        f'Moedas não suportadas: {sorted(moedas_inesperadas)}. Esperadas: BRL e USD.'
    )

print(f'Moedas validadas: {sorted(moedas_encontradas)}')


Moedas validadas: ['BRL', 'USD']


In [6]:
tipos_esperados = {
    'deposito',
    'pagamento',
    'saque',
    'transferencia_enviada',
    'transferencia_recebida',
}
tipos_inesperados = set(df_operacoes_tratado['tipo'].unique()) - tipos_esperados

if tipos_inesperados:
    raise ValueError(f'Tipos não suportados: {sorted(tipos_inesperados)}')

print('O tipo saque é aceito normalmente nesta base.')


O tipo saque é aceito normalmente nesta base.


In [7]:
df_operacoes_tratado['valor_brl'] = df_operacoes_tratado['valor'].astype(float)
mascara_usd = df_operacoes_tratado['moeda'].eq('USD')
df_operacoes_tratado.loc[mascara_usd, 'valor_brl'] = (
    df_operacoes_tratado.loc[mascara_usd, 'valor'].astype(float) * taxa_cambio_usd_brl
)

assert df_operacoes_tratado['valor_brl'].notna().all()
print('valor e moeda originais foram preservados; valor_brl foi criado.')


valor e moeda originais foram preservados; valor_brl foi criado.


## Regra 1 - Fracionamento

Cada sinalização da Regra 1 representa um evento `cliente_id + data`, e não uma operação individual. As seis operações únicas sem data permanecem na base tratada e participam de volume, contagens e mediana, mas não participam deste agrupamento temporal.


In [8]:
MINIMO_OPERACOES_FRACIONAMENTO = 3
LIMITE_SOMA_FRACIONAMENTO = 50_000
LIMITE_VALOR_INDIVIDUAL = 20_000

df_operacoes_com_data = df_operacoes_tratado.loc[
    df_operacoes_tratado['data'].notna()
].copy()

resumo_fracionamento = (
    df_operacoes_com_data.groupby(['cliente_id', 'data'], as_index=False)
    .agg(
        quantidade_operacoes=('id', 'size'),
        soma_valor_brl=('valor_brl', 'sum'),
        maior_valor_individual_brl=('valor_brl', 'max'),
    )
)

resumo_fracionamento['flag_fracionamento'] = (
    (resumo_fracionamento['quantidade_operacoes'] >= MINIMO_OPERACOES_FRACIONAMENTO)
    & (resumo_fracionamento['soma_valor_brl'] > LIMITE_SOMA_FRACIONAMENTO)
    & (resumo_fracionamento['maior_valor_individual_brl'] < LIMITE_VALOR_INDIVIDUAL)
)

eventos_fracionamento = resumo_fracionamento.loc[
    resumo_fracionamento['flag_fracionamento']
].copy()

# A flag por operacao serve apenas para inspecao; o ranking conta o evento agrupado.
df_operacoes_tratado = df_operacoes_tratado.merge(
    resumo_fracionamento[['cliente_id', 'data', 'flag_fracionamento']],
    on=['cliente_id', 'data'],
    how='left',
    validate='many_to_one',
)
df_operacoes_tratado['flag_fracionamento'] = (
    df_operacoes_tratado['flag_fracionamento'].fillna(False).astype(bool)
)


## Regra 2 - Valor atípico

A mediana e a quantidade de operações são calculadas por cliente. Cada operação que supera cinco vezes a mediana, para clientes com ao menos quatro operações, conta como uma sinalização.


In [9]:
MINIMO_OPERACOES_VALOR_ATIPICO = 4
MULTIPLICADOR_VALOR_ATIPICO = 5

perfil_valor_por_cliente = (
    df_operacoes_tratado.groupby('cliente_id', as_index=False)
    .agg(
        quantidade_operacoes_cliente=('id', 'size'),
        mediana_valor_brl=('valor_brl', 'median'),
    )
)

df_operacoes_tratado = df_operacoes_tratado.merge(
    perfil_valor_por_cliente,
    on='cliente_id',
    how='left',
    validate='many_to_one',
)
df_operacoes_tratado['limite_valor_atipico_brl'] = (
    MULTIPLICADOR_VALOR_ATIPICO * df_operacoes_tratado['mediana_valor_brl']
)
df_operacoes_tratado['flag_valor_atipico'] = (
    (df_operacoes_tratado['quantidade_operacoes_cliente'] >= MINIMO_OPERACOES_VALOR_ATIPICO)
    & (df_operacoes_tratado['valor_brl'] > df_operacoes_tratado['limite_valor_atipico_brl'])
)


## Ranking por cliente

O ranking soma eventos de fracionamento e operações atípicas, sem usar LLM. Em caso de mesmo total de sinalizações, o maior volume total em BRL vem primeiro.


In [10]:
sinais_regra_1_por_cliente = (
    eventos_fracionamento.groupby('cliente_id')
    .size()
    .rename('sinais_regra_1')
)
sinais_regra_2_por_cliente = (
    df_operacoes_tratado.loc[df_operacoes_tratado['flag_valor_atipico']]
    .groupby('cliente_id')
    .size()
    .rename('sinais_regra_2')
)

ranking_clientes = (
    df_operacoes_tratado.groupby('cliente_id', as_index=False)
    .agg(volume_total_brl=('valor_brl', 'sum'))
    .merge(sinais_regra_1_por_cliente, on='cliente_id', how='left')
    .merge(sinais_regra_2_por_cliente, on='cliente_id', how='left')
)
ranking_clientes[['sinais_regra_1', 'sinais_regra_2']] = (
    ranking_clientes[['sinais_regra_1', 'sinais_regra_2']].fillna(0).astype(int)
)
ranking_clientes['total_sinalizacoes'] = (
    ranking_clientes['sinais_regra_1'] + ranking_clientes['sinais_regra_2']
)
ranking_clientes = ranking_clientes.sort_values(
    ['total_sinalizacoes', 'volume_total_brl'],
    ascending=[False, False],
).reset_index(drop=True)

top_10_clientes = ranking_clientes.head(10).copy()
top_10_clientes.insert(0, 'posicao', range(1, len(top_10_clientes) + 1))


In [11]:
assert len(df_operacoes_bruto) == 322
assert df_operacoes_bruto['data'].isna().sum() == 7
assert len(df_operacoes_tratado) == 317
assert df_operacoes_tratado['cliente_id'].nunique() == 30
assert df_operacoes_tratado['data'].isna().sum() == 6
assert not moedas_inesperadas
assert (
    ranking_clientes['total_sinalizacoes']
    == ranking_clientes['sinais_regra_1'] + ranking_clientes['sinais_regra_2']
).all()

print('Validações da Parte A concluídas com sucesso.')


Validações da Parte A concluídas com sucesso.


In [12]:
quantidade_eventos_fracionamento = len(eventos_fracionamento)
quantidade_operacoes_atipicas = int(df_operacoes_tratado['flag_valor_atipico'].sum())
quantidade_clientes_sinalizados = int((ranking_clientes['total_sinalizacoes'] > 0).sum())

print(f'Eventos de fracionamento: {quantidade_eventos_fracionamento}')
print(f'Operações atípicas: {quantidade_operacoes_atipicas}')
print(f'Clientes com ao menos uma sinalização: {quantidade_clientes_sinalizados}')


Eventos de fracionamento: 4
Operações atípicas: 21
Clientes com ao menos uma sinalização: 17


In [13]:
colunas_top_10 = [
    'posicao',
    'cliente_id',
    'sinais_regra_1',
    'sinais_regra_2',
    'total_sinalizacoes',
    'volume_total_brl',
]
print(top_10_clientes[colunas_top_10].to_string(index=False))


 posicao cliente_id  sinais_regra_1  sinais_regra_2  total_sinalizacoes  volume_total_brl
       1    CLI-014               0               3                   3         80629.990
       2    CLI-023               0               2                   2        148535.016
       3    CLI-028               0               2                   2         88750.800
       4    CLI-013               0               2                   2         81730.990
       5    CLI-005               0               2                   2         64742.660
       6    CLI-026               0               2                   2         54729.280
       7    CLI-001               0               2                   2         47947.810
       8    CLI-029               1               0                   1        191385.766
       9    CLI-017               1               0                   1        121391.370
      10    CLI-030               0               1                   1        117780.886


In [14]:
empates_no_top_10 = top_10_clientes.loc[
    top_10_clientes['total_sinalizacoes'].duplicated(keep=False),
    [
        'posicao',
        'cliente_id',
        'total_sinalizacoes',
        'volume_total_brl',
    ],
].copy()

if empates_no_top_10.empty:
    print('Não houve empate em total de sinalizações no Top 10.')
else:
    print('Empates em total de sinalizações no Top 10; volume_total_brl definiu a ordem:')
    print(empates_no_top_10.to_string(index=False))


Empates em total de sinalizações no Top 10; volume_total_brl definiu a ordem:
 posicao cliente_id  total_sinalizacoes  volume_total_brl
       2    CLI-023                   2        148535.016
       3    CLI-028                   2         88750.800
       4    CLI-013                   2         81730.990
       5    CLI-005                   2         64742.660
       6    CLI-026                   2         54729.280
       7    CLI-001                   2         47947.810
       8    CLI-029                   1        191385.766
       9    CLI-017                   1        121391.370
      10    CLI-030                   1        117780.886


## Reaproveitamento do Nível 1

Foram mantidos os mesmos limites, critérios e cálculos em Pandas. A diferença desta etapa é que o ranking usa uma linha por evento de fracionamento e uma linha por operação atípica, consolidando ambas as contagens por cliente.
